# Detecció d'opinions
La primera part de la pràctica 2 consta de la detecció d'opinions classificra-les coma  negatives o positives
El primer que s'ha realitzat es importar les dades que utilitzarem com a corpus

In [2]:
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews as mr


[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


## Algoritmes d'aprenentatge supervisat

### Preparació de les dades

La primera part de la preparació de les dades consisteix a netejar-les. Aquest procés inclou dos passos principals:

Primer, normalitzem el text posant-lo tot en minúscules i eliminant qualsevol caràcter que no sigui una lletra de l'alfabet (a-z), com ara números, signes de puntuació o símbols especials. Això fara que el model tracti "genial!", "genial," com paraules diferents.

Segon, lematitzem el text, és a dir, reduïm cada paraula a la seva forma base o lema. Per exemple, "running", "runs" i "ran" es converteixen totes en "run". Gràcies a això, el vocabulari és molt més compacte i el model pot aprendre millor les relacions entre paraules.

In [7]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text_string(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 1
    ]

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashve\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


El corpus "movie_review" ja compta amb les etiquetes "pos" i "neg" per a opinions positives i negatives respectivament. Per preparar les dades, hem separat aquestes etiquetes i hem dividit el corpus en train i test, amb un 20% del corpus per al test i un 80% per a l'entrenament dels models.
A més, s'aplica la funció feta anteriorment, que deixa el text net per poder entrenar els models adequadament.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Separar les etiquetes ("pos" i "neg") que fan referència a la categoria
documents = []
labels = []

for categoria in mr.categories():
    for fileid in mr.fileids(categoria):
        raw_text = mr.raw(fileid)                 
        clean_text = clean_text_string(raw_text)  

        if clean_text:                            
            documents.append(clean_text)
            labels.append(categoria)

# Si no fem això a vegades s'afageixen opinions buides i genera errors, 
# per tant, només si hi ha text s'afageix a les dades

# Divisió del corpus train/test

X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42
)


A continuació, com que les dades són strings, les convertim a números, ja que els models que hem escollit per classificar només funcionen amb valors numèrics. Per fer això, utilitzem TfidfVectorizer, que converteix el text en vectors numèrics.

Hem d'escollir l'hiperparàmetre max_features, que fa referència a la mida del vocabulari que utilitzarà el model. Per escollir-lo, cal tenir en compte que si és massa petit es perdrà molta informació, i si és massa gran augmentarà considerablement el cost computacional.

Per això hem fet una experimentació amb diversos valors de l'hiperparàmetre, utilitzant MultinomialNB com a model de referència per avaluar l'accuracy en cada cas. Hem escollit aquest model perquè és l'adequat per a vectors TF-IDF, ja que cada paraula pot aparèixer amb pesos decimals de valor variable, a diferència de BernoulliNB, que només contempla si una paraula apareix o no.

In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

valors = [1000, 3000, 5000, 8000, 11457, 15000, 20000, 34369]

for n in valors:
    vec = TfidfVectorizer(max_features=n)
    X_tr = vec.fit_transform(X_train)
    X_te = vec.transform(X_test)
    
    model = MultinomialNB()
    model.fit(X_tr, y_train)
    acc = accuracy_score(y_test, model.predict(X_te))
    
    print(f"max_features={n:6d} → accuracy: {acc:.4f}")

max_features=  1000 → accuracy: 0.7625
max_features=  3000 → accuracy: 0.7875
max_features=  5000 → accuracy: 0.8050
max_features=  8000 → accuracy: 0.8125
max_features= 11457 → accuracy: 0.8075
max_features= 15000 → accuracy: 0.8125
max_features= 20000 → accuracy: 0.8125
max_features= 34369 → accuracy: 0.8125


In [17]:
vectorizer = TfidfVectorizer(max_features=15000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

### SVM (Super Vector Machine)

### XGBoost

### Regressió Logística